# RetailPulse — 02 · Data Cleaning

**Phase 3 deliverable** · produces the analysis-ready tables in `dataset/cleaned/`

---

## 1 · Introduction and cleaning objectives

Phase 1 *measured* 17 data quality issues. This notebook *acts* on them. The two
were kept apart deliberately: you cannot choose a sensible fix before you know
the size and the cause of a problem, and a cleaning step with no recorded
reason is indistinguishable from a bug.

### The governing principle: no row is ever deleted

This is the most important rule in the notebook, and it is worth saying plainly
because it is the opposite of what most tutorials do.

A typical cleaning script calls `dropna()` and `drop_duplicates()` and moves
on. Both are destructive and both are usually wrong here:

- `dropna()` on `orders` would delete 2,965 orders whose only sin is that they
  were never delivered. They are real orders with real revenue.
- `drop_duplicates(subset='review_id')` would delete 814 real review scores
  attached to real orders, because `review_id` repeats for a reason that has
  nothing to do with duplication.

Instead, this notebook **flags** problems in new columns and **scopes
exclusions to the specific metric** that needs them. An order missing a
delivery date is excluded from *delivery-time averages* and kept everywhere
else. The result is that every number later in the project can be traced back
to the exact rows behind it.

### What this notebook does

| | |
|---|---|
| **Reads** | the 9 raw CSVs in `dataset/raw/` — **read-only, never modified** |
| **Produces** | 9 cleaned tables in `dataset/cleaned/` |
| **Fixes** | correct data types, English category names, derived analytical columns |
| **Flags** | impossible dates, payment anomalies, missing-delivery cases |
| **Deletes** | **nothing** |

### Objectives, in order

1. Load the raw data without letting pandas corrupt it by guessing types
2. Convert each column to its correct type, deliberately
3. Decide and record a rule for every column containing missing values
4. Investigate duplicates before touching them
5. Re-confirm that keys and relationships still hold after cleaning
6. Repair the category translation gap without losing products
7. Validate date sequences and flag the impossible ones
8. Check the numeric columns for impossible values
9. Build the derived columns the analysis and dashboards need
10. Export, then validate the exported files against the raw inputs

## 2 · Load raw datasets

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW     = PROJECT_ROOT / "dataset" / "raw"
CLEANED = PROJECT_ROOT / "dataset" / "cleaned"
CLEANED.mkdir(parents=True, exist_ok=True)

assert RAW.exists(), f"Raw folder missing: {RAW}"
print("raw     :", RAW)
print("cleaned :", CLEANED)
print("pandas", pd.__version__)

raw     : /Users/manish/Desktop/RetailPulse/dataset/raw
cleaned : /Users/manish/Desktop/RetailPulse/dataset/cleaned
pandas 3.1.1


### Reusable helpers

Six small functions used throughout. Defining them once keeps the cleaning
steps readable and means a validation rule is written in exactly one place.

In [2]:
# --------------------------------------------------------------- loading ---
RAW_FILES = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
    "geolocation": "olist_geolocation_dataset.csv",
}

# Only the translation file carries a UTF-8 BOM. Read without utf-8-sig, its
# first column name keeps an invisible character and every join to it silently
# matches nothing - no error, just empty results.
ENCODINGS = {"translation": "utf-8-sig"}


def load_raw(key: str) -> pd.DataFrame:
    """Load one raw CSV as all-text. dtype=str stops pandas guessing types and
    destroying zip-code leading zeros or turning IDs into floats."""
    return pd.read_csv(RAW / RAW_FILES[key], dtype=str,
                       encoding=ENCODINGS.get(key, "utf-8"))


# ------------------------------------------------------------ validation ---
def check_pk(df: pd.DataFrame, cols, label: str) -> dict:
    """A primary key must be unique across all rows AND never null. Both."""
    cols = [cols] if isinstance(cols, str) else list(cols)
    sub = df[cols]
    n, distinct = len(df), len(sub.drop_duplicates())
    nulls = int(sub.isna().any(axis=1).sum())
    return {"key": label, "rows": n, "distinct": distinct, "nulls": nulls,
            "valid": bool(distinct == n and nulls == 0)}


def check_fk(child, ckey, parent, pkey, label) -> dict:
    """Count child rows whose key has no matching parent row (orphans)."""
    cvals = child[ckey].dropna()
    orphans = int((~cvals.isin(set(parent[pkey].dropna()))).sum())
    return {"relationship": label, "child_rows": len(child), "orphans": orphans,
            "ok": bool(orphans == 0)}


def nulls_table(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Null count and percentage for every column that has any."""
    s = df.isna().sum()
    s = s[s > 0]
    return pd.DataFrame({"table": name, "column": s.index, "nulls": s.values,
                         "null_pct": (s.values / len(df) * 100).round(2)})


def to_numeric_strict(s: pd.Series, name: str) -> pd.Series:
    """Convert to numeric and FAIL LOUDLY if any value could not be parsed.
    A silent NaN from a bad conversion is how corrupted money columns happen."""
    out = pd.to_numeric(s, errors="coerce")
    introduced = int(out.isna().sum() - s.isna().sum())
    if introduced:
        raise ValueError(f"{name}: {introduced} values failed numeric conversion")
    return out


def to_datetime_strict(s: pd.Series, name: str) -> pd.Series:
    """Same idea for timestamps."""
    out = pd.to_datetime(s, errors="coerce")
    introduced = int(out.isna().sum() - s.isna().sum())
    if introduced:
        raise ValueError(f"{name}: {introduced} values failed datetime conversion")
    return out


# ---------------------------------------------------------------- ledger ---
# Every decision is appended here and printed in section 14, so the summary is
# generated from what the code actually did rather than written by hand.
LEDGER = []

def log(issue, table, action, rows, detail):
    LEDGER.append({"issue": issue, "table": table, "action": action,
                   "rows_affected": rows, "detail": detail})

print("helpers defined")

helpers defined


In [3]:
raw = {k: load_raw(k) for k in RAW_FILES}
RAW_COUNTS = {k: len(v) for k, v in raw.items()}

print(f"{'table':<14}{'rows':>12}  {'cols':>5}")
for k, v in raw.items():
    print(f"{k:<14}{len(v):>12,}  {v.shape[1]:>5}")
print(f"{'TOTAL':<14}{sum(RAW_COUNTS.values()):>12,}")

table                 rows   cols
orders              99,441      8
order_items        112,650      7
payments           103,886      5
reviews             99,224      7
customers           99,441      5
products            32,951      9
sellers              3,095      4
translation             71      2
geolocation      1,000,163      5
TOTAL            1,550,922


## 3 · Validate initial row counts

Before changing anything, confirm the inputs are exactly what Phase 1 profiled.
If a file has changed underneath us, everything downstream is suspect — so this
cell **raises an error** rather than printing a warning. A validation that can
be ignored is not a validation.

In [4]:
EXPECTED = {
    "orders": 99441, "order_items": 112650, "payments": 103886,
    "reviews": 99224, "customers": 99441, "products": 32951,
    "sellers": 3095, "translation": 71, "geolocation": 1000163,
}

mismatch = {k: (RAW_COUNTS[k], EXPECTED[k])
            for k in EXPECTED if RAW_COUNTS[k] != EXPECTED[k]}
if mismatch:
    raise AssertionError(f"Raw row counts differ from Phase 1: {mismatch}")

print("All 9 raw tables match the Phase 1 profiled row counts exactly.")
print(f"Total input rows: {sum(RAW_COUNTS.values()):,}")
print()
print("Note: the reviews file has 104,720 physical LINES but 99,224 RECORDS —")
print("3,852 comments contain newlines. The CSV parser is authoritative here.")

All 9 raw tables match the Phase 1 profiled row counts exactly.
Total input rows: 1,550,922

Note: the reviews file has 104,720 physical LINES but 99,224 RECORDS —
3,852 comments contain newlines. The CSV parser is authoritative here.


## 4 · Correct data types

Everything arrived as text. Now each column is converted deliberately.

**Three rules, and the reason for each:**

| Rule | Why |
|---|---|
| IDs stay **text** | They are 32-character hex strings, not numbers. Converting loses leading characters and invites float rounding |
| Zip prefixes stay **text** | `01037` as a number becomes `1037` — a different, wrong place |
| Money becomes **float**, dates become **datetime** | So they can be summed, averaged and compared |

Conversions use the strict helpers, which raise an error if any value fails to
parse. Silent `NaN` from a failed conversion is exactly how a money column gets
quietly corrupted.

In [5]:
clean = {}

# ----- orders -------------------------------------------------------------
o = raw["orders"].copy()
ORDER_DATES = ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"]
for c in ORDER_DATES:
    o[c] = to_datetime_strict(o[c], f"orders.{c}")
clean["orders"] = o

# ----- order_items --------------------------------------------------------
oi = raw["order_items"].copy()
oi["order_item_id"] = to_numeric_strict(oi.order_item_id, "order_item_id").astype("int16")
oi["shipping_limit_date"] = to_datetime_strict(oi.shipping_limit_date, "shipping_limit_date")
for c in ["price", "freight_value"]:
    oi[c] = to_numeric_strict(oi[c], f"order_items.{c}").astype("float64")
clean["order_items"] = oi

# ----- payments -----------------------------------------------------------
pay = raw["payments"].copy()
pay["payment_sequential"] = to_numeric_strict(pay.payment_sequential, "payment_sequential").astype("int16")
pay["payment_installments"] = to_numeric_strict(pay.payment_installments, "payment_installments").astype("int16")
pay["payment_value"] = to_numeric_strict(pay.payment_value, "payment_value").astype("float64")
clean["payments"] = pay

# ----- reviews ------------------------------------------------------------
rv = raw["reviews"].copy()
rv["review_score"] = to_numeric_strict(rv.review_score, "review_score").astype("int8")
for c in ["review_creation_date", "review_answer_timestamp"]:
    rv[c] = to_datetime_strict(rv[c], f"reviews.{c}")
clean["reviews"] = rv

# ----- products: fix the source misspellings (issue Q16) ------------------
pr = raw["products"].copy()
RENAMES = {"product_name_lenght": "product_name_length",
           "product_description_lenght": "product_description_length"}
pr = pr.rename(columns=RENAMES)
for c in ["product_name_length", "product_description_length", "product_photos_qty",
          "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    pr[c] = to_numeric_strict(pr[c], f"products.{c}").astype("Float64")
clean["products"] = pr
log("Q16", "products", "renamed misspelled source columns", 2,
    "product_name_lenght -> product_name_length; product_description_lenght -> product_description_length")

# ----- customers / sellers: IDs and zips deliberately left as text --------
clean["customers"] = raw["customers"].copy()
clean["sellers"] = raw["sellers"].copy()

# ----- geolocation --------------------------------------------------------
ge = raw["geolocation"].copy()
for c in ["geolocation_lat", "geolocation_lng"]:
    ge[c] = to_numeric_strict(ge[c], f"geolocation.{c}").astype("float64")
clean["geolocation"] = ge

clean["translation"] = raw["translation"].copy()

print("Type conversion complete — no value failed to parse.\n")
print("Spot-check that zip codes kept their leading zeros:")
z = clean["customers"].customer_zip_code_prefix
lead = z[z.str.startswith("0")]
print(f"  customer zips starting with '0': {len(lead):,}  e.g. {lead.iloc[0]!r}")
print(f"  dtype: {z.dtype}  <- text, as intended (numeric would give 1037 not '01037')")
print()
print("orders dtypes after conversion:")
print(clean["orders"].dtypes.to_string())

Type conversion complete — no value failed to parse.

Spot-check that zip codes kept their leading zeros:
  customer zips starting with '0': 23,995  e.g. '09790'
  dtype: str  <- text, as intended (numeric would give 1037 not '01037')

orders dtypes after conversion:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]


## 5 · Handle missing values

**Nothing is filled in.** Every missing value here is either meaningful or
harmless, and inventing data to replace it would be the single worst thing this
notebook could do.

The specific temptation worth naming: `order_delivered_customer_date` is
missing for 2,965 orders, and `order_estimated_delivery_date` is right there
and always populated. Copying the estimate into the actual would make the data
*look* complete — and would silently make on-time delivery 100%, because every
order would then arrive exactly as promised. The metric would be destroyed and
the chart would look perfect. **The gap is kept.**

In [6]:
null_summary = pd.concat(
    [nulls_table(df, name) for name, df in clean.items()], ignore_index=True
).sort_values("null_pct", ascending=False).reset_index(drop=True)
display(null_summary)

DECISIONS = {
    "review_comment_title":          ("KEEP NULL", "Optional field. A missing title means no title."),
    "review_comment_message":        ("KEEP NULL", "Optional field. 41k customers did write one."),
    "order_delivered_customer_date": ("KEEP NULL", "Correct for undelivered orders. Flagged where status says 'delivered' (Q01)."),
    "order_delivered_carrier_date":  ("KEEP NULL", "Correct — the order never shipped."),
    "order_approved_at":             ("KEEP NULL", "Approval lag is not a planned KPI (Q15)."),
    "product_category_name":         ("LABEL 'unknown'", "610 products keep their revenue; only the label is supplied (Q04)."),
    "product_name_length":           ("KEEP NULL", "Listing metadata, unused by any KPI."),
    "product_description_length":    ("KEEP NULL", "Listing metadata, unused by any KPI."),
    "product_photos_qty":            ("KEEP NULL", "Listing metadata, unused by any KPI."),
    "product_weight_g":              ("KEEP NULL", "2 rows. No KPI uses it (Q13)."),
    "product_length_cm":             ("KEEP NULL", "2 rows."),
    "product_height_cm":             ("KEEP NULL", "2 rows."),
    "product_width_cm":              ("KEEP NULL", "2 rows."),
}
null_summary["decision"] = null_summary.column.map(lambda c: DECISIONS.get(c, ("REVIEW", ""))[0])
null_summary["reason"]   = null_summary.column.map(lambda c: DECISIONS.get(c, ("", "not yet decided"))[1])

print("\nDecision for every column containing missing values:")
display(null_summary[["table", "column", "nulls", "null_pct", "decision", "reason"]])

undecided = null_summary[null_summary.decision == "REVIEW"]
assert undecided.empty, f"Undocumented null columns: {list(undecided.column)}"
filled = (null_summary.decision != "KEEP NULL").sum()
print(f"\nColumns with nulls: {len(null_summary)}  |  kept as NULL: "
      f"{(null_summary.decision == 'KEEP NULL').sum()}  |  labelled: {filled}")
print("Values invented to fill a gap: 0")
log("Q04/Q13/Q15", "multiple", "documented a decision for every null column",
    int(null_summary.nulls.sum()), "No value was imputed. One column is labelled 'unknown'.")

,table,column,nulls,null_pct
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_name_length,610,1.85
5,products,product_description_length,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01



Decision for every column containing missing values:


,table,column,nulls,null_pct,decision,reason
0,reviews,review_comment_title,87656,88.34,KEEP NULL,Optional field. A missing title means no title.
1,reviews,review_comment_message,58247,58.70,KEEP NULL,Optional field. 41k customers did write one.
2,orders,order_delivered_customer_date,2965,2.98,KEEP NULL,Correct for undelivered orders. Flagged where ...
3,products,product_category_name,610,1.85,LABEL 'unknown',610 products keep their revenue; only the labe...
4,products,product_name_length,610,1.85,KEEP NULL,"Listing metadata, unused by any KPI."
5,products,product_description_length,610,1.85,KEEP NULL,"Listing metadata, unused by any KPI."
6,products,product_photos_qty,610,1.85,KEEP NULL,"Listing metadata, unused by any KPI."
7,orders,order_delivered_carrier_date,1783,1.79,KEEP NULL,Correct — the order never shipped.
8,orders,order_approved_at,160,0.16,KEEP NULL,Approval lag is not a planned KPI (Q15).
9,products,product_weight_g,2,0.01,KEEP NULL,2 rows. No KPI uses it (Q13).



Columns with nulls: 13  |  kept as NULL: 12  |  labelled: 1
Values invented to fill a gap: 0


## 6 · Investigate duplicate records

Three different things get called "duplicate". They need different answers, so
they are measured separately.

In [7]:
print("A · Exact duplicate rows (every column identical)")
for name, df in clean.items():
    d = int(df.duplicated().sum())
    if d:
        print(f"    {name:<14}{d:>10,}  ({d/len(df)*100:.1f}%)")
print("    (no other table has any)")
print()

print("B · The geolocation duplicates — benign, and here is the proof")
ge = clean["geolocation"]
print(f"    rows {len(ge):,} | exact duplicates {int(ge.duplicated().sum()):,} "
      f"| distinct zip prefixes {ge.geolocation_zip_code_prefix.nunique():,}")
print(f"    rows per prefix: mean {len(ge)/ge.geolocation_zip_code_prefix.nunique():.1f}, "
      f"max {ge.geolocation_zip_code_prefix.value_counts().max():,}")
print("    A zip prefix covers an AREA, so many coordinate readings share one prefix.")
print("    These are repeated readings, not corrupted records.")
print()

print("C · Repeated review_id — investigate BEFORE acting (issue Q02)")
rv = clean["reviews"]
dup_ids = rv.review_id.value_counts()
dup_ids = dup_ids[dup_ids > 1]
print(f"    review_id values appearing more than once : {len(dup_ids):,}")
print(f"    rows involved                             : {int(dup_ids.sum()):,}")
print(f"    exact duplicate ROWS in reviews           : {int(rv.duplicated().sum()):,}")
ex = dup_ids.index[0]
print(f"\n    Example — review_id {ex[:14]}… appears {dup_ids.iloc[0]} times:")
print(rv.loc[rv.review_id == ex,
             ["review_id", "order_id", "review_score", "review_creation_date"]]
      .to_string(index=False))
same_score = rv[rv.review_id.isin(dup_ids.index)].groupby("review_id").review_score.nunique()
print(f"\n    Of the {len(dup_ids):,} repeated ids, how many carry DIFFERENT scores "
      f"across their rows? {int((same_score > 1).sum())}")
print("    Every repeated id carries the SAME score on every row, but a DIFFERENT order_id.")
print()
print("    Conclusion: one survey response covering several orders from one purchase")
print("    event. The rows are not duplicates of each other - each ties a real score")
print("    to a different real order. drop_duplicates(subset='review_id') would delete")
print(f"    {int(dup_ids.sum() - len(dup_ids)):,} genuine order-to-score links.")
print()
print("    DECISION: keep every row. Use the composite key (review_id, order_id).")

A · Exact duplicate rows (every column identical)


    geolocation      261,831  (26.2%)
    (no other table has any)

B · The geolocation duplicates — benign, and here is the proof
    rows 1,000,163 | exact duplicates 261,831 | distinct zip prefixes 19,015


    rows per prefix: mean 52.6, max 1,146
    A zip prefix covers an AREA, so many coordinate readings share one prefix.
    These are repeated readings, not corrupted records.

C · Repeated review_id — investigate BEFORE acting (issue Q02)
    review_id values appearing more than once : 789
    rows involved                             : 1,603
    exact duplicate ROWS in reviews           : 0

    Example — review_id c444278834184f… appears 3 times:
                       review_id                         order_id  review_score review_creation_date
c444278834184f72b1484dfe47de7f97 df56136b8031ecd28e200bb18e6ddb2e             5           2017-02-08
c444278834184f72b1484dfe47de7f97 566f53fc7d36fa366f7e221468121877             5           2017-02-08
c444278834184f72b1484dfe47de7f97 8557dabbdacec1a9e250f5e70afe1eab             5           2017-02-08

    Of the 789 repeated ids, how many carry DIFFERENT scores across their rows? 0
    Every repeated id carries the SAME score on every row,

In [8]:
# Confirm the composite key actually works before relying on it.
pk_checks = pd.DataFrame([
    check_pk(clean["reviews"], "review_id", "reviews.review_id"),
    check_pk(clean["reviews"], ["review_id", "order_id"], "reviews.(review_id, order_id)"),
    check_pk(clean["order_items"], ["order_id", "order_item_id"], "order_items.(order_id, order_item_id)"),
    check_pk(clean["payments"], ["order_id", "payment_sequential"], "payments.(order_id, payment_sequential)"),
])
display(pk_checks)

assert not pk_checks.loc[0, "valid"], "review_id unexpectedly unique"
assert pk_checks.loc[1, "valid"], "composite review key failed"
assert pk_checks.loc[2, "valid"] and pk_checks.loc[3, "valid"], "composite key failed"
print("Composite keys confirmed valid. Rows deleted in this section: 0")
log("Q02", "reviews", "kept all rows; adopted (review_id, order_id) composite key",
    int(dup_ids.sum()), "Repeated ids carry the same score against DIFFERENT orders - not duplicates.")
log("Q12", "geolocation", "duplicates confirmed benign; aggregated later in section 11",
    int(clean["geolocation"].duplicated().sum()), "Multiple coordinate readings per zip prefix area.")

,key,rows,distinct,nulls,valid
0,reviews.review_id,99224,98410,0,False
1,"reviews.(review_id, order_id)",99224,99224,0,True
2,"order_items.(order_id, order_item_id)",112650,112650,0,True
3,"payments.(order_id, payment_sequential)",103886,103886,0,True


Composite keys confirmed valid. Rows deleted in this section: 0


## 7 · Validate primary and foreign keys

Re-running the Phase 1 checks on the *converted* data. Type conversion can
break a key — a zip code silently turned numeric stops matching its text
counterpart elsewhere — so the relationships are re-tested rather than assumed
to have survived.

In [9]:
pk_all = pd.DataFrame([
    check_pk(clean["orders"],      "order_id",                          "orders.order_id"),
    check_pk(clean["customers"],   "customer_id",                       "customers.customer_id"),
    check_pk(clean["customers"],   "customer_unique_id",                "customers.customer_unique_id"),
    check_pk(clean["products"],    "product_id",                        "products.product_id"),
    check_pk(clean["sellers"],     "seller_id",                         "sellers.seller_id"),
    check_pk(clean["translation"], "product_category_name",             "translation.product_category_name"),
    check_pk(clean["order_items"], ["order_id", "order_item_id"],       "order_items.(order_id, order_item_id)"),
    check_pk(clean["payments"],    ["order_id", "payment_sequential"],  "payments.(order_id, payment_sequential)"),
    check_pk(clean["reviews"],     ["review_id", "order_id"],           "reviews.(review_id, order_id)"),
])
display(pk_all)

# customer_unique_id is EXPECTED to fail - it identifies the person, and 3,345
# of these rows belong to someone who ordered before. That failure is the basis
# of all retention analysis.
expected_invalid = {"customers.customer_unique_id"}
unexpected = pk_all[(~pk_all.valid) & (~pk_all.key.isin(expected_invalid))]
assert unexpected.empty, f"Key broke during cleaning: {list(unexpected.key)}"
print("All primary keys intact after type conversion.")
print("customers.customer_unique_id is correctly NOT unique — it is the person, "
      "and 3,345 rows are repeat buyers.")

,key,rows,distinct,nulls,valid
0,orders.order_id,99441,99441,0,True
1,customers.customer_id,99441,99441,0,True
2,customers.customer_unique_id,99441,96096,0,False
3,products.product_id,32951,32951,0,True
4,sellers.seller_id,3095,3095,0,True
5,translation.product_category_name,71,71,0,True
6,"order_items.(order_id, order_item_id)",112650,112650,0,True
7,"payments.(order_id, payment_sequential)",103886,103886,0,True
8,"reviews.(review_id, order_id)",99224,99224,0,True


All primary keys intact after type conversion.
customers.customer_unique_id is correctly NOT unique — it is the person, and 3,345 rows are repeat buyers.


In [10]:
fk_all = pd.DataFrame([
    check_fk(clean["orders"],      "customer_id", clean["customers"], "customer_id", "orders → customers"),
    check_fk(clean["order_items"], "order_id",    clean["orders"],    "order_id",    "order_items → orders"),
    check_fk(clean["payments"],    "order_id",    clean["orders"],    "order_id",    "payments → orders"),
    check_fk(clean["reviews"],     "order_id",    clean["orders"],    "order_id",    "reviews → orders"),
    check_fk(clean["order_items"], "product_id",  clean["products"],  "product_id",  "order_items → products"),
    check_fk(clean["order_items"], "seller_id",   clean["sellers"],   "seller_id",   "order_items → sellers"),
])
display(fk_all)
assert fk_all.orphans.sum() == 0, "Orphan rows appeared during cleaning"
print("Zero orphans across all six relationships — referential integrity preserved.")
print("(products → translation is handled separately in section 8; it is the one")
print(" relationship with genuine gaps.)")

,relationship,child_rows,orphans,ok
0,orders → customers,99441,0,True
1,order_items → orders,112650,0,True
2,payments → orders,103886,0,True
3,reviews → orders,99224,0,True
4,order_items → products,112650,0,True
5,order_items → sellers,112650,0,True


Zero orphans across all six relationships — referential integrity preserved.
(products → translation is handled separately in section 8; it is the one
 relationship with genuine gaps.)


## 8 · Handle untranslated and missing product categories

The only broken relationship in the dataset. `products` holds 73 distinct
category names; the translation lookup covers 71.

An `INNER JOIN` here would delete 13 products **without any error message** —
the quiet kind of bug that produces a plausible-looking but wrong category
report. A `LEFT JOIN` keeps every product and leaves the gap visible so it can
be filled on purpose.

In [11]:
pr, tr = clean["products"], clean["translation"]
prod_cats = set(pr.product_category_name.dropna())
trans_cats = set(tr.product_category_name)
missing = sorted(prod_cats - trans_cats)

print(f"distinct categories in products      : {len(prod_cats)}")
print(f"categories in the translation table  : {len(trans_cats)}")
print(f"MISSING from the translation table   : {len(missing)}")
for c in missing:
    print(f"    '{c}'  →  {int((pr.product_category_name == c).sum())} products")
print(f"products with a NULL category        : {int(pr.product_category_name.isna().sum()):,}")

# Supply the two missing translations explicitly, rather than letting them
# fall through to a null and disappear from category reports.
MANUAL_TRANSLATIONS = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers",
}
extra = pd.DataFrame({
    "product_category_name": list(MANUAL_TRANSLATIONS),
    "product_category_name_english": list(MANUAL_TRANSLATIONS.values()),
})
tr_full = pd.concat([tr, extra], ignore_index=True)
clean["translation"] = tr_full
print(f"\ntranslation table: {len(tr)} → {len(tr_full)} rows (2 added by hand)")
log("Q03", "translation", "added 2 missing English category names", 13,
    "pc_gamer, portateis_cozinha_e_preparadores_de_alimentos — LEFT JOIN keeps all 13 products")

distinct categories in products      : 73
categories in the translation table  : 71
MISSING from the translation table   : 2
    'pc_gamer'  →  3 products
    'portateis_cozinha_e_preparadores_de_alimentos'  →  10 products
products with a NULL category        : 610

translation table: 71 → 73 rows (2 added by hand)


In [12]:
# LEFT JOIN, then label the remaining nulls 'unknown'.
before = len(pr)
pr = pr.merge(tr_full, on="product_category_name", how="left")
assert len(pr) == before, "The category join changed the row count"

pr["product_category_name_english"] = pr.product_category_name_english.fillna("unknown")
pr["product_category_name"] = pr.product_category_name.fillna("unknown")
clean["products"] = pr

print(f"products rows: {before:,} → {len(pr):,}  (unchanged — nothing was dropped)")
print(f"products still without an English name: "
      f"{int((pr.product_category_name_english == 'unknown').sum()):,} "
      f"(the 610 that never had a category at all)")
print()
print("Top categories by product count:")
print(pr.product_category_name_english.value_counts().head(8).to_string())

# Prove the inner-join alternative would have silently lost rows.
inner_n = len(clean["products"][["product_id"]].merge(
    raw["products"][["product_id", "product_category_name"]], on="product_id")
    .merge(tr, on="product_category_name", how="inner"))
print(f"\nIf an INNER JOIN had been used instead: {inner_n:,} products would survive")
print(f"  → {before - inner_n:,} products silently deleted (13 untranslated + 610 null).")
log("Q04", "products", "labelled null categories 'unknown' instead of dropping", 610,
    "These products carry real revenue; dropping them would understate category totals.")

products rows: 32,951 → 32,951  (unchanged — nothing was dropped)
products still without an English name: 610 (the 610 that never had a category at all)

Top categories by product count:
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411

If an INNER JOIN had been used instead: 32,328 products would survive
  → 623 products silently deleted (13 untranslated + 610 null).


## 9 · Validate date and timestamp sequences

A date column can be perfectly formatted and still describe something
impossible. These checks test the *logic* of the event order, not the format.

In [13]:
o = clean["orders"]
seq_checks = {
    "approved before purchased":            o.order_approved_at < o.order_purchase_timestamp,
    "delivered before purchased":           o.order_delivered_customer_date < o.order_purchase_timestamp,
    "delivered before carrier handoff":     o.order_delivered_customer_date < o.order_delivered_carrier_date,
    "estimated delivery before purchase":   o.order_estimated_delivery_date < o.order_purchase_timestamp,
    "carrier handoff before purchase":      o.order_delivered_carrier_date < o.order_purchase_timestamp,
}
rows = [{"check": k, "violations": int(v.sum())} for k, v in seq_checks.items()]
display(pd.DataFrame(rows))

# Flag the impossible sequences; do NOT rewrite the timestamps. We cannot know
# the true value, and inventing one would be worse than recording the problem.
bad_seq = seq_checks["delivered before carrier handoff"].fillna(False)
o["has_date_anomaly"] = bad_seq
o["date_anomaly_note"] = np.where(bad_seq, "delivered_before_carrier_handoff", "")
clean["orders"] = o

print(f"\nOrders flagged with has_date_anomaly = True: {int(bad_seq.sum())}")
print("Timestamps were NOT altered. The true values are unknowable from this data,")
print("so the anomaly is recorded in a column and carried forward visibly.")
print("\nMost likely cause: a logistics scanning lag — the courier recorded the")
print("handoff after the parcel had already moved.")
log("Q08", "orders", "flagged impossible date sequence (no timestamp altered)",
    int(bad_seq.sum()), "delivered_customer_date earlier than delivered_carrier_date")

,check,violations
0,approved before purchased,0
1,delivered before purchased,0
2,delivered before carrier handoff,23
3,estimated delivery before purchase,0
4,carrier handoff before purchase,166



Orders flagged with has_date_anomaly = True: 23
Timestamps were NOT altered. The true values are unknowable from this data,
so the anomaly is recorded in a column and carried forward visibly.

Most likely cause: a logistics scanning lag — the courier recorded the
handoff after the parcel had already moved.


In [14]:
# Date coverage, and the stable analysis window.
o = clean["orders"]
monthly = o.order_purchase_timestamp.dt.to_period("M").value_counts().sort_index()
print(f"Purchase window: {o.order_purchase_timestamp.min()} → {o.order_purchase_timestamp.max()}")
print(f"Distinct months present: {len(monthly)}")
print()
print("Orders per month across the whole range:")
for p, n in monthly.items():
    flag = "   <-- sparse" if n < 200 else ""
    print(f"    {p}  {n:>6,}{flag}")

full_range = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
absent = [str(p) for p in full_range if p not in monthly.index]
print(f"\nMonths with NO rows at all: {absent if absent else 'none'}")
print("2016 is not merely thin — it is discontinuous. November 2016 is absent")
print("entirely, so any line chart spanning it would draw straight across a gap.")

Purchase window: 2016-09-04 21:15:19 → 2018-10-17 17:30:18
Distinct months present: 25

Orders per month across the whole range:
    2016-09       4   <-- sparse
    2016-10     324
    2016-12       1   <-- sparse
    2017-01     800
    2017-02   1,780
    2017-03   2,682
    2017-04   2,404
    2017-05   3,700
    2017-06   3,245
    2017-07   4,026
    2017-08   4,331
    2017-09   4,285
    2017-10   4,631
    2017-11   7,544
    2017-12   5,673
    2018-01   7,269
    2018-02   6,728
    2018-03   7,211
    2018-04   6,939
    2018-05   6,873
    2018-06   6,167
    2018-07   6,292
    2018-08   6,512
    2018-09      16   <-- sparse
    2018-10       4   <-- sparse

Months with NO rows at all: ['2016-11']
2016 is not merely thin — it is discontinuous. November 2016 is absent
entirely, so any line chart spanning it would draw straight across a gap.


In [15]:
# in_stable_window flags the 20 complete months. It is a FLAG, not a filter:
# the rows stay, and each metric decides for itself whether to use it.
STABLE_START, STABLE_END = "2017-01", "2018-08"
om = o.order_purchase_timestamp.dt.to_period("M").astype(str)
o["in_stable_window"] = ((om >= STABLE_START) & (om <= STABLE_END))
clean["orders"] = o

n_stable = int(o.in_stable_window.sum())
print(f"Stable window {STABLE_START} … {STABLE_END}")
print(f"  orders inside  : {n_stable:,}  ({n_stable/len(o)*100:.2f}%)")
print(f"  orders outside : {len(o)-n_stable:,}  ({(len(o)-n_stable)/len(o)*100:.2f}%)")
print()
print("Trend, growth and cohort analysis use this flag. Grand totals do not —")
print("they use every row. No order is deleted either way.")
log("Q06/Q07", "orders", "added in_stable_window flag (no rows removed)", len(o) - n_stable,
    f"{STABLE_START}..{STABLE_END}; edge months hold 0.35% of orders and 2016-11 is absent")

Stable window 2017-01 … 2018-08
  orders inside  : 99,092  (99.65%)
  orders outside : 349  (0.35%)

Trend, growth and cohort analysis use this flag. Grand totals do not —
they use every row. No order is deleted either way.


## 10 · Validate numerical columns

Checking the money columns for values that cannot be real.

In [16]:
oi, pay = clean["order_items"], clean["payments"]
num = {
    "order_items.price":           oi.price,
    "order_items.freight_value":   oi.freight_value,
    "payments.payment_value":      pay.payment_value,
    "reviews.review_score":        clean["reviews"].review_score,
}
summary = pd.DataFrame({
    k: {"count": int(s.count()), "min": s.min(), "median": s.median(),
        "mean": s.mean(), "max": s.max(),
        "zeros": int((s == 0).sum()), "negatives": int((s < 0).sum())}
    for k, s in num.items()
}).T
display(summary)

assert (oi.price > 0).all(), "A non-positive item price appeared"
assert (oi.freight_value >= 0).all(), "A negative freight value appeared"
assert clean["reviews"].review_score.between(1, 5).all(), "review_score outside 1-5"
print("No negative money anywhere. All review scores within 1–5.")
print(f"\nfreight_value == 0 : {int((oi.freight_value == 0).sum()):,} items — "
      "plausible (free-shipping promotions). Kept as-is.")

,count,min,median,mean,max,zeros,negatives
order_items.price,"112,650.00",0.85,74.99,120.65,"6,735.00",0.00,0.00
order_items.freight_value,"112,650.00",0.00,16.26,19.99,409.68,383.00,0.00
payments.payment_value,"103,886.00",0.00,100.00,154.10,"13,664.08",9.00,0.00
reviews.review_score,"99,224.00",1.00,5.00,4.09,5.00,0.00,0.00


No negative money anywhere. All review scores within 1–5.

freight_value == 0 : 383 items — plausible (free-shipping promotions). Kept as-is.


In [17]:
# Payment anomalies: flag, retain, never silently alter.
pay["payment_anomaly"] = ""
zero_val = pay.payment_value == 0
zero_inst = pay.payment_installments == 0
undef_type = pay.payment_type == "not_defined"

pay.loc[zero_val, "payment_anomaly"] = "zero_payment_value"
pay.loc[zero_inst, "payment_anomaly"] = (pay.loc[zero_inst, "payment_anomaly"]
                                         .str.cat(["zero_installments"] * int(zero_inst.sum()), sep="|")
                                         .str.strip("|"))
pay.loc[undef_type, "payment_anomaly"] = (pay.loc[undef_type, "payment_anomaly"]
                                          .str.cat(["undefined_payment_type"] * int(undef_type.sum()), sep="|")
                                          .str.strip("|"))

# payment_installments = 0 is not a valid count. Store a corrected value in a
# NEW column and keep the original untouched, so the assumption is auditable.
pay["payment_installments_clean"] = pay.payment_installments.replace(0, 1)
pay["payment_type_clean"] = pay.payment_type.replace("not_defined", "unknown")
clean["payments"] = pay

print(f"zero payment_value        : {int(zero_val.sum())}  → flagged, retained (Q09)")
print(f"zero payment_installments : {int(zero_inst.sum())}  → flagged; _clean column reads 1 (Q10)")
print(f"payment_type not_defined  : {int(undef_type.sum())}  → flagged; _clean column reads 'unknown' (Q11)")
print(f"\nrows carrying any payment anomaly: {int((pay.payment_anomaly != '').sum())}")
print("Original payment_installments and payment_type columns are unchanged.")
for q, n, d in [("Q09", int(zero_val.sum()), "payment_value = 0 flagged and retained"),
                ("Q10", int(zero_inst.sum()), "installments 0 → 1 in a NEW column; original kept"),
                ("Q11", int(undef_type.sum()), "'not_defined' → 'unknown' in a NEW column; original kept")]:
    log(q, "payments", "flagged, retained", n, d)

zero payment_value        : 9  → flagged, retained (Q09)
zero payment_installments : 2  → flagged; _clean column reads 1 (Q10)
payment_type not_defined  : 3  → flagged; _clean column reads 'unknown' (Q11)

rows carrying any payment anomaly: 11
Original payment_installments and payment_type columns are unchanged.


## 11 · Create derived analytical columns

New columns the analysis needs. **No original column is overwritten** — every
derived value goes into a new column alongside the source it came from.

### The rule that matters most here: aggregate before joining

`order_items` has 112,650 rows for 98,666 orders. `payments` has 103,886 rows
for 99,440 orders. Reviews has 99,224 rows for 98,673 orders.

Join any of those straight onto `orders` and the order rows **multiply** — and
every duplicated row carries a duplicated revenue figure. Measured on this
dataset, joining items to payments inflates revenue by **+4.54%**; joining
items to reviews moves it **−0.47%** (an inner join both duplicates *and*
drops). Nothing errors. The totals are simply wrong.

So each child table is collapsed to **one row per order first**, and only then
joined.

In [18]:
# ---- 1. Revenue and freight per order, from the ITEM grain ---------------
order_money = (clean["order_items"]
               .groupby("order_id", as_index=False)
               .agg(item_count=("order_item_id", "size"),
                    order_revenue=("price", "sum"),
                    order_freight=("freight_value", "sum"),
                    n_sellers=("seller_id", "nunique")))
print(f"order_items {len(clean['order_items']):,} rows → {len(order_money):,} orders")

# ---- 2. Reviews per order, from the REVIEW grain -------------------------
order_reviews = (clean["reviews"]
                 .groupby("order_id", as_index=False)
                 .agg(review_score_avg=("review_score", "mean"),
                      review_count=("review_id", "size")))
print(f"reviews     {len(clean['reviews']):,} rows → {len(order_reviews):,} orders")

# ---- 3. Payments per order, from the PAYMENT grain -----------------------
order_pay = (clean["payments"]
             .groupby("order_id", as_index=False)
             .agg(payment_total=("payment_value", "sum"),
                  payment_count=("payment_sequential", "size"),
                  max_installments=("payment_installments_clean", "max")))
print(f"payments    {len(clean['payments']):,} rows → {len(order_pay):,} orders")
print("\nEach is now one row per order, so the joins below cannot fan out.")

order_items 112,650 rows → 98,666 orders
reviews     99,224 rows → 98,673 orders


payments    103,886 rows → 99,440 orders

Each is now one row per order, so the joins below cannot fan out.


In [19]:
o = clean["orders"].copy()
before = len(o)

o = (o.merge(clean["customers"][["customer_id", "customer_unique_id",
                                 "customer_state", "customer_city",
                                 "customer_zip_code_prefix"]],
             on="customer_id", how="left")
       .merge(order_money,   on="order_id", how="left")
       .merge(order_reviews, on="order_id", how="left")
       .merge(order_pay,     on="order_id", how="left"))

# The guard that catches fan-out. If any join multiplied rows, stop here.
assert len(o) == before, f"FAN-OUT: orders went {before:,} → {len(o):,}"
print(f"orders after 4 joins: {len(o):,} (unchanged — no fan-out) ✓")

# ---- calendar columns ----------------------------------------------------
o["order_date"]  = o.order_purchase_timestamp.dt.date
o["order_month"] = o.order_purchase_timestamp.dt.to_period("M").astype(str)
o["order_year"]  = o.order_purchase_timestamp.dt.year.astype("int16")

# ---- status flags --------------------------------------------------------
EXCLUDED_FROM_REVENUE = ["canceled", "unavailable"]
o["is_delivered"]     = (o.order_status == "delivered")
o["revenue_eligible"] = (~o.order_status.isin(EXCLUDED_FROM_REVENUE))
o["has_items"]        = o.item_count.notna()
o["has_review"]       = o.review_count.notna()

# Orders with no line items genuinely earned zero, so 0 is the true value here
# — this is not imputation.
o["item_count"]    = o.item_count.fillna(0).astype("int16")
o["order_revenue"] = o.order_revenue.fillna(0.0)
o["order_freight"] = o.order_freight.fillna(0.0)
o["n_sellers"]     = o.n_sellers.fillna(0).astype("int16")
o["review_count"]  = o.review_count.fillna(0).astype("int16")
print("calendar columns and status flags added")

orders after 4 joins: 99,441 (unchanged — no fan-out) ✓


calendar columns and status flags added


In [20]:
# ---- delivery metrics, from EXACT timestamps -----------------------------
# delivery_days and delay_days are computed to full precision. is_late is
# derived from the UNROUNDED difference and only then is the displayed value
# rounded. Rounding first would reclassify an order late by 4 minutes as
# on-time - a real bug caught earlier in this project.
has_actual = o.order_delivered_customer_date.notna()

delivery_exact = (o.order_delivered_customer_date - o.order_purchase_timestamp).dt.total_seconds() / 86400
delay_exact    = (o.order_delivered_customer_date - o.order_estimated_delivery_date).dt.total_seconds() / 86400

o["delivery_days"] = delivery_exact.round(4)
o["delay_days"]    = delay_exact.round(4)
o["is_late"]       = pd.Series(pd.NA, index=o.index, dtype="boolean")
o.loc[has_actual, "is_late"] = (delay_exact[has_actual] > 0)

# Q01: 8 orders claim 'delivered' but carry no delivery date. They are kept in
# revenue and order counts, and excluded ONLY from delivery timing.
o["delivery_measurable"] = has_actual
q01 = int((o.is_delivered & ~has_actual).sum())
o["has_delivery_anomaly"] = o.is_delivered & ~has_actual

clean["orders"] = o
print(f"orders with a real delivery date   : {int(has_actual.sum()):,}")
print(f"status 'delivered' but NO date     : {q01}   ← flagged (Q01), kept in revenue")
print(f"late deliveries                    : {int(o.is_late.sum()):,}")
print(f"on-time or early                   : {int((~o.is_late.fillna(False) & has_actual).sum()):,}")
print(f"on-time rate                       : {(1 - o.loc[has_actual,'is_late'].mean())*100:.2f}%")
print()
print("Sanity: how many orders would a ROUNDED comparison misclassify?")
rounded_late = (delay_exact[has_actual].round(2) > 0)
print(f"  exact: {int((delay_exact[has_actual] > 0).sum()):,} late | "
      f"rounded-first: {int(rounded_late.sum()):,} late | "
      f"difference: {int((delay_exact[has_actual] > 0).sum() - rounded_late.sum())}")
log("Q01", "orders", "flagged 'delivered' with no date; excluded from timing only", q01,
    "Delivery KPI denominator is 96,476, not 96,478. Rows kept in revenue and counts.")

orders with a real delivery date   : 96,476
status 'delivered' but NO date     : 8   ← flagged (Q01), kept in revenue
late deliveries                    : 7,827
on-time or early                   : 88,649
on-time rate                       : 91.89%

Sanity: how many orders would a ROUNDED comparison misclassify?
  exact: 7,827 late | rounded-first: 7,821 late | difference: 6


In [21]:
# ---- item grain: English category + order context ------------------------
oi = clean["order_items"].copy()
before = len(oi)
oi = (oi.merge(clean["products"][["product_id", "product_category_name",
                                  "product_category_name_english"]],
               on="product_id", how="left")
        .merge(clean["sellers"][["seller_id", "seller_state", "seller_city"]],
               on="seller_id", how="left")
        .merge(clean["orders"][["order_id", "order_status", "order_date",
                                "order_month", "order_year", "revenue_eligible",
                                "customer_state"]],
               on="order_id", how="left"))
assert len(oi) == before, f"FAN-OUT in order_items: {before:,} → {len(oi):,}"
oi["item_total"] = oi.price + oi.freight_value
clean["order_items"] = oi
print(f"order_items: {len(oi):,} rows (unchanged) with category, seller and order context ✓")

# ---- geolocation: collapse to one point per zip prefix -------------------
ge = clean["geolocation"]
geo_clean = (ge.groupby("geolocation_zip_code_prefix", as_index=False)
             .agg(geolocation_lat=("geolocation_lat", "median"),
                  geolocation_lng=("geolocation_lng", "median"),
                  geolocation_city=("geolocation_city", lambda s: s.mode().iloc[0]),
                  geolocation_state=("geolocation_state", lambda s: s.mode().iloc[0]),
                  readings=("geolocation_lat", "size")))
clean["geolocation"] = geo_clean
print(f"geolocation: {len(ge):,} readings → {len(geo_clean):,} zip prefixes "
      f"({(1-len(geo_clean)/len(ge))*100:.1f}% smaller)")
print("  median lat/lng is used, not mean: a median ignores the occasional")
print("  wildly wrong coordinate instead of being dragged towards it.")
log("Q12", "geolocation", "aggregated to one row per zip prefix",
    int(len(ge) - len(geo_clean)), "median lat/lng, modal city/state; reference data only")

order_items: 112,650 rows (unchanged) with category, seller and order context ✓


geolocation: 1,000,163 readings → 19,015 zip prefixes (98.1% smaller)
  median lat/lng is used, not mean: a median ignores the occasional
  wildly wrong coordinate instead of being dragged towards it.


## 12 · Export cleaned datasets

Nine tables written to `dataset/cleaned/`. CSV is chosen over Parquet on
purpose: Phase 4 loads these into MySQL and Phase 6 into Power BI, and both
read CSV natively.

In [22]:
EXPORTS = {
    "orders_clean.csv":              clean["orders"],
    "order_items_clean.csv":         clean["order_items"],
    "order_payments_clean.csv":      clean["payments"],
    "order_reviews_clean.csv":       clean["reviews"],
    "customers_clean.csv":           clean["customers"],
    "products_clean.csv":            clean["products"],
    "sellers_clean.csv":             clean["sellers"],
    "category_translation_clean.csv":clean["translation"],
    "geolocation_clean.csv":         clean["geolocation"],
}

manifest = []
for fname, df in EXPORTS.items():
    path = CLEANED / fname
    # Default quoting handles the 3,852 review comments containing newlines:
    # fields with embedded newlines are quoted, so the file reads back correctly.
    df.to_csv(path, index=False)
    mb = path.stat().st_size / 1024**2
    manifest.append({"file": fname, "rows": len(df), "columns": df.shape[1],
                     "size_mb": round(mb, 2)})
    print(f"  {fname:<32}{len(df):>9,} rows × {df.shape[1]:>2} cols   {mb:>6.1f} MB")

mf = pd.DataFrame(manifest)
print(f"\nTotal: {mf.rows.sum():,} rows across {len(mf)} files, {mf.size_mb.sum():.1f} MB")

  orders_clean.csv                   99,441 rows × 36 cols     32.6 MB


  order_items_clean.csv             112,650 rows × 18 cols     24.4 MB
  order_payments_clean.csv          103,886 rows ×  8 cols      6.7 MB


  order_reviews_clean.csv            99,224 rows ×  7 cols     13.3 MB
  customers_clean.csv                99,441 rows ×  5 cols      8.2 MB
  products_clean.csv                 32,951 rows × 10 cols      3.1 MB
  sellers_clean.csv                   3,095 rows ×  4 cols      0.2 MB
  category_translation_clean.csv         73 rows ×  2 cols      0.0 MB


  geolocation_clean.csv              19,015 rows ×  6 cols      1.1 MB

Total: 569,776 rows across 9 files, 89.6 MB


In [23]:
# Read every file back and confirm it round-trips. Writing a file is not proof
# that it can be read - especially with embedded newlines in review text.
print("Round-trip check — re-reading each exported file:\n")
roundtrip = []
for fname, original in EXPORTS.items():
    back = pd.read_csv(CLEANED / fname, dtype=str)
    ok = len(back) == len(original) and list(back.columns) == list(original.columns)
    roundtrip.append({"file": fname, "written": len(original), "read_back": len(back),
                      "cols_match": list(back.columns) == list(original.columns), "ok": ok})
    print(f"  {'OK ' if ok else 'XX '} {fname:<32}{len(original):>9,} → {len(back):>9,}")

rt = pd.DataFrame(roundtrip)
assert rt.ok.all(), f"Round-trip failed: {list(rt[~rt.ok].file)}"
print("\nAll 9 files re-read with identical row counts and columns ✓")

rv_back = pd.read_csv(CLEANED / "order_reviews_clean.csv", dtype=str)
emb = rv_back.review_comment_message.fillna("").str.contains("\n").sum()
print(f"Reviews with embedded newlines survived the round trip: {int(emb):,} "
      f"(expected 3,852-ish; CSV quoting worked)")

Round-trip check — re-reading each exported file:



  OK  orders_clean.csv                   99,441 →    99,441
  OK  order_items_clean.csv             112,650 →   112,650
  OK  order_payments_clean.csv          103,886 →   103,886


  OK  order_reviews_clean.csv            99,224 →    99,224
  OK  customers_clean.csv                99,441 →    99,441
  OK  products_clean.csv                 32,951 →    32,951
  OK  sellers_clean.csv                   3,095 →     3,095
  OK  category_translation_clean.csv         73 →        73
  OK  geolocation_clean.csv              19,015 →    19,015

All 9 files re-read with identical row counts and columns ✓


Reviews with embedded newlines survived the round trip: 3,852 (expected 3,852-ish; CSV quoting worked)


## 13 · Final data quality validation

The gate. Every check compares the cleaned output against the raw input, and
**asserts** — so a regression stops the notebook rather than producing a
quietly wrong file.

In [24]:
checks = []
def V(name, passed, detail):
    checks.append({"check": name, "result": "PASS" if passed else "FAIL", "detail": detail})
    return passed

# --- row counts -----------------------------------------------------------
row_map = {"orders": "orders", "order_items": "order_items", "payments": "payments",
           "reviews": "reviews", "customers": "customers", "products": "products",
           "sellers": "sellers"}
for key, ck in row_map.items():
    V(f"row count preserved: {key}", len(clean[ck]) == RAW_COUNTS[key],
      f"raw {RAW_COUNTS[key]:,} → cleaned {len(clean[ck]):,}")
V("translation grew by exactly 2", len(clean["translation"]) == RAW_COUNTS["translation"] + 2,
  f"raw {RAW_COUNTS['translation']} → cleaned {len(clean['translation'])} (2 manual additions)")
V("geolocation intentionally aggregated",
  len(clean["geolocation"]) == 19015,
  f"raw {RAW_COUNTS['geolocation']:,} readings → {len(clean['geolocation']):,} zip prefixes")

# --- keys -----------------------------------------------------------------
for d in [check_pk(clean["orders"], "order_id", "orders.order_id"),
          check_pk(clean["customers"], "customer_id", "customers.customer_id"),
          check_pk(clean["products"], "product_id", "products.product_id"),
          check_pk(clean["sellers"], "seller_id", "sellers.seller_id"),
          check_pk(clean["order_items"], ["order_id","order_item_id"], "order_items composite"),
          check_pk(clean["payments"], ["order_id","payment_sequential"], "payments composite"),
          check_pk(clean["reviews"], ["review_id","order_id"], "reviews composite")]:
    V(f"PK valid: {d['key']}", d["valid"], f"{d['distinct']:,} distinct of {d['rows']:,}")

# --- foreign keys ---------------------------------------------------------
for d in [check_fk(clean["orders"], "customer_id", clean["customers"], "customer_id", "orders → customers"),
          check_fk(clean["order_items"], "order_id", clean["orders"], "order_id", "order_items → orders"),
          check_fk(clean["payments"], "order_id", clean["orders"], "order_id", "payments → orders"),
          check_fk(clean["reviews"], "order_id", clean["orders"], "order_id", "reviews → orders")]:
    V(f"FK intact: {d['relationship']}", d["ok"], f"{d['orphans']} orphans")

# --- money reconciliation -------------------------------------------------
raw_gmv = pd.to_numeric(raw["order_items"].price).sum()
cln_gmv = clean["order_items"].price.sum()
V("GMV preserved exactly", abs(raw_gmv - cln_gmv) < 0.01,
  f"raw {raw_gmv:,.2f} vs cleaned {cln_gmv:,.2f}")
raw_fr = pd.to_numeric(raw["order_items"].freight_value).sum()
V("freight preserved exactly", abs(raw_fr - clean["order_items"].freight_value.sum()) < 0.01,
  f"raw {raw_fr:,.2f}")
raw_pay = pd.to_numeric(raw["payments"].payment_value).sum()
V("payment total preserved", abs(raw_pay - clean["payments"].payment_value.sum()) < 0.01,
  f"raw {raw_pay:,.2f}")
# order_revenue must sum to GMV: proof the aggregation lost nothing
V("order-level revenue reconciles to item GMV",
  abs(clean["orders"].order_revenue.sum() - cln_gmv) < 0.01,
  f"orders {clean['orders'].order_revenue.sum():,.2f} vs items {cln_gmv:,.2f}")

# --- counts against Phase 1 ----------------------------------------------
o = clean["orders"]
V("delivered orders = 96,478", int(o.is_delivered.sum()) == 96478, f"{int(o.is_delivered.sum()):,}")
V("orders with delivery date = 96,476", int(o.delivery_measurable.sum()) == 96476,
  f"{int(o.delivery_measurable.sum()):,}")
V("late orders = 7,827", int(o.is_late.sum()) == 7827, f"{int(o.is_late.sum()):,}")
V("distinct customers = 96,096", o.customer_unique_id.nunique() == 96096,
  f"{o.customer_unique_id.nunique():,}")
V("orders with items = 98,666", int(o.has_items.sum()) == 98666, f"{int(o.has_items.sum()):,}")
V("orders with review = 98,673", int(o.has_review.sum()) == 98673, f"{int(o.has_review.sum()):,}")
V("review mean = 4.0864", abs(clean["reviews"].review_score.mean() - 4.0864) < 0.0001,
  f"{clean['reviews'].review_score.mean():.4f}")

# --- no silent data loss --------------------------------------------------
V("no NULL order_id anywhere", clean["orders"].order_id.notna().all(), "0 nulls")
V("every product has an English category label",
  clean["products"].product_category_name_english.notna().all(),
  "610 labelled 'unknown', 0 left null")
V("no negative money", bool((clean["order_items"].price > 0).all()
                            and (clean["order_items"].freight_value >= 0).all()), "")

results = pd.DataFrame(checks)
display(results)
failed = results[results.result == "FAIL"]
print(f"\nChecks run: {len(results)}   PASS: {int((results.result=='PASS').sum())}   FAIL: {len(failed)}")
if len(failed):
    display(failed)
    raise AssertionError(f"{len(failed)} validation check(s) failed")
print("\nALL VALIDATION CHECKS PASSED ✓")

,check,result,detail
0,row count preserved: orders,PASS,"raw 99,441 → cleaned 99,441"
1,row count preserved: order_items,PASS,"raw 112,650 → cleaned 112,650"
2,row count preserved: payments,PASS,"raw 103,886 → cleaned 103,886"
3,row count preserved: reviews,PASS,"raw 99,224 → cleaned 99,224"
4,row count preserved: customers,PASS,"raw 99,441 → cleaned 99,441"
5,row count preserved: products,PASS,"raw 32,951 → cleaned 32,951"
6,row count preserved: sellers,PASS,"raw 3,095 → cleaned 3,095"
7,translation grew by exactly 2,PASS,raw 71 → cleaned 73 (2 manual additions)
8,geolocation intentionally aggregated,PASS,"raw 1,000,163 readings → 19,015 zip prefixes"
9,PK valid: orders.order_id,PASS,"99,441 distinct of 99,441"



Checks run: 34   PASS: 34   FAIL: 0

ALL VALIDATION CHECKS PASSED ✓


## 14 · Cleaning summary

In [25]:
print("=" * 80)
print("ROW COUNTS — raw vs cleaned")
print("=" * 80)
print(f"{'table':<16}{'raw':>12}{'cleaned':>12}{'delta':>10}  note")
notes = {
    "translation": "+2 manual English category names (Q03)",
    "geolocation": "aggregated to one row per zip prefix (Q12)",
}
for key in RAW_FILES:
    ck = clean[key]
    d = len(ck) - RAW_COUNTS[key]
    print(f"{key:<16}{RAW_COUNTS[key]:>12,}{len(ck):>12,}{d:>+10,}  {notes.get(key,'unchanged')}")
print()
print(f"Rows DELETED because they were 'bad': 0")
print(f"Values IMPUTED / invented:            0")

ROW COUNTS — raw vs cleaned
table                    raw     cleaned     delta  note
orders                99,441      99,441        +0  unchanged
order_items          112,650     112,650        +0  unchanged
payments             103,886     103,886        +0  unchanged
reviews               99,224      99,224        +0  unchanged
customers             99,441      99,441        +0  unchanged
products              32,951      32,951        +0  unchanged
sellers                3,095       3,095        +0  unchanged
translation               71          73        +2  +2 manual English category names (Q03)
geolocation        1,000,163      19,015  -981,148  aggregated to one row per zip prefix (Q12)

Rows DELETED because they were 'bad': 0
Values IMPUTED / invented:            0


In [26]:
ledger = pd.DataFrame(LEDGER)
print("=" * 80)
print("CLEANING DECISION LEDGER (generated from what the code actually did)")
print("=" * 80)
display(ledger)

print("\nDerived columns added to orders_clean:")
derived = ["order_date", "order_month", "order_year", "order_revenue", "order_freight",
           "item_count", "n_sellers", "delivery_days", "delay_days", "is_late",
           "is_delivered", "revenue_eligible", "has_items", "has_review",
           "in_stable_window", "delivery_measurable", "has_date_anomaly",
           "has_delivery_anomaly", "review_score_avg", "review_count",
           "payment_total", "payment_count", "max_installments"]
present = [c for c in derived if c in clean["orders"].columns]
for i in range(0, len(present), 3):
    print("   " + "  ".join(f"{c:<24}" for c in present[i:i+3]))
print(f"\norders_clean: {clean['orders'].shape[1]} columns "
      f"({len(raw['orders'].columns)} original + {clean['orders'].shape[1]-len(raw['orders'].columns)} derived/joined)")

CLEANING DECISION LEDGER (generated from what the code actually did)


,issue,table,action,rows_affected,detail
0,Q16,products,renamed misspelled source columns,2,product_name_lenght -> product_name_length; pr...
1,Q04/Q13/Q15,multiple,documented a decision for every null column,153259,No value was imputed. One column is labelled '...
2,Q02,reviews,"kept all rows; adopted (review_id, order_id) c...",1603,Repeated ids carry the same score against DIFF...
3,Q12,geolocation,duplicates confirmed benign; aggregated later ...,261831,Multiple coordinate readings per zip prefix area.
4,Q03,translation,added 2 missing English category names,13,"pc_gamer, portateis_cozinha_e_preparadores_de_..."
5,Q04,products,labelled null categories 'unknown' instead of ...,610,These products carry real revenue; dropping th...
6,Q08,orders,flagged impossible date sequence (no timestamp...,23,delivered_customer_date earlier than delivered...
7,Q06/Q07,orders,added in_stable_window flag (no rows removed),349,2017-01..2018-08; edge months hold 0.35% of or...
8,Q09,payments,"flagged, retained",9,payment_value = 0 flagged and retained
9,Q10,payments,"flagged, retained",2,installments 0 → 1 in a NEW column; original kept



Derived columns added to orders_clean:
   order_date                order_month               order_year              
   order_revenue             order_freight             item_count              
   n_sellers                 delivery_days             delay_days              
   is_late                   is_delivered              revenue_eligible        
   has_items                 has_review                in_stable_window        
   delivery_measurable       has_date_anomaly          has_delivery_anomaly    
   review_score_avg          review_count              payment_total           
   payment_count             max_installments        

orders_clean: 36 columns (8 original + 28 derived/joined)


In [27]:
print("=" * 80)
print("HEADLINE FIGURES FROM THE CLEANED DATA")
print("=" * 80)
o, oi = clean["orders"], clean["order_items"]
elig_items = oi[oi.revenue_eligible]
elig_orders = o[o.revenue_eligible & o.has_items]
hdr = {
    "Total orders":                      f"{len(o):,}",
    "Orders with line items":            f"{int(o.has_items.sum()):,}",
    "Delivered orders":                  f"{int(o.is_delivered.sum()):,}",
    "Distinct customers (people)":       f"{o.customer_unique_id.nunique():,}",
    "Product revenue — all statuses":    f"{oi.price.sum():,.2f} BRL",
    "Product revenue — eligible":        f"{elig_items.price.sum():,.2f} BRL",
    "Freight charged":                   f"{oi.freight_value.sum():,.2f} BRL",
    "AOV (eligible orders with items)":  f"{elig_items.price.sum()/len(elig_orders):,.2f} BRL",
    "Median order value (eligible)":     f"{elig_orders.order_revenue.median():,.2f} BRL",
    "Avg delivery time":                 f"{o.delivery_days.mean():.2f} days",
    "Median delivery time":              f"{o.delivery_days.median():.2f} days",
    "On-time delivery rate":             f"{(1-o.loc[o.delivery_measurable,'is_late'].mean())*100:.2f}%",
    "Avg review score (review grain)":   f"{clean['reviews'].review_score.mean():.4f}",
    "Cancellation rate":                 f"{(o.order_status=='canceled').mean()*100:.4f}%",
}
for k, v in hdr.items():
    print(f"  {k:<36}{v:>22}")
print()
print("Every figure above is reproducible: rerun this notebook and it recomputes")
print("from dataset/raw/, which was never modified.")
print()
print("NEXT: Phase 4 — MySQL schema design and 25-30 analytical queries.")

HEADLINE FIGURES FROM THE CLEANED DATA
  Total orders                                        99,441
  Orders with line items                              98,666
  Delivered orders                                    96,478
  Distinct customers (people)                         96,096
  Product revenue — all statuses           13,591,643.70 BRL
  Product revenue — eligible               13,494,400.74 BRL
  Freight charged                           2,251,909.54 BRL
  AOV (eligible orders with items)                137.42 BRL
  Median order value (eligible)                    86.90 BRL
  Avg delivery time                               12.56 days
  Median delivery time                            10.22 days
  On-time delivery rate                               91.89%
  Avg review score (review grain)                     4.0864
  Cancellation rate                                  0.6285%

Every figure above is reproducible: rerun this notebook and it recomputes
from dataset/raw/, which was nev